# Diniz Ch.7 — RLS tracking of a time-varying plant

Problems 15–16 的 Markov 係數模型及 λ=0.995/0.9。呈現 lag/memory 理論、實測 MSE 和 MSD。

來源：`Diniz-Chap7-SignalPred.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Tracking generator and factories

In [ ]:
from adsp.experiments import diniz

n_samples, n_trials, seed = 3000, 50, 2026
make_data = partial(data.tracking, n_samples, initial_weights=diniz.TRACKING_WEIGHTS, coefficient_memory=0.999,
                    coefficient_noise_var=0.033, noise_var=0.01)
models = {
  f"A priori LRLS λ={lam}": partial(af.APrioriLatticeRLS, filter_order=7, lam=lam, epsilon=0.1)
  for lam in (0.995, 0.9)
}
models.update({f"Direct RLS λ={lam}": partial(af.RLS, filter_order=7, lam=lam, delta=10.0) for lam in (0.995, 0.9)})
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 3. Tracking MSE and theory

In [ ]:
plot.learning(results, window=64, title="Time-varying plant tracking")
plot.show()
training.summary(results, tail=2500)
for lam in (0.995, 0.9):
  print(f"λ={lam}: approximate lag/memory theory", diniz.tracking_theory(lam))
plot.learning(results, quantity="msd", window=64, title="Direct RLS: physical plant coefficient MSD")
plot.show()
plot.coefficients(results["A priori LRLS λ=0.9"], title="Lattice feedforward coefficients: basis dependent")
plot.show()